In [15]:
import os

print(os.getcwd())
print(os.listdir())

/Users/saraelbachouti/Desktop/UCV-Churn
['eda5.ipynb', 'primermodelosimple.ipynb', '.DS_Store', 'LICENSE', 'requirements.txt', 'segundomodelo.ipynb', 'config', 'Dockerfile', 'eda4.ipynb', 'edatercerenfoque.ipynb', 'models', 'README.md', 'modelofinal.ipynb', '.gitignore', 'edasegundoenfoque.ipynb', 'UCV-Churn', 'dataset_limpio_churn.csv', '.git', 'processed', 'data', 'edaprimerenfoque.ipynb', 'notebooks', 'reports', 'src']


In [16]:
# ============================================================
# MODELO XGBOOST CORRECTO
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ============================================================
# 1. CARGAR DATASET
# ============================================================

df_model = pd.read_csv("dataset_limpio_churn.csv")

dataset = df_model.copy()

print("Shape inicial:", dataset.shape)


# ============================================================
# 2. FECHAS
# ============================================================

if "fecha" in dataset.columns:

    dataset["fecha"] = pd.to_datetime(
        dataset["fecha"],
        errors="coerce"
    )

    if "cliente_id" in dataset.columns:

        dataset = dataset.sort_values(
            ["cliente_id", "fecha"]
        )


# ============================================================
# 3. FEATURES TEMPORALES
# ============================================================

if (
    "cliente_id" in dataset.columns
    and
    "importe_total" in dataset.columns
):

    dataset["media_importe_3m"] = (
        dataset
        .groupby("cliente_id")["importe_total"]
        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).mean()
        )
    )

    dataset["importe_mes_anterior"] = (
        dataset
        .groupby("cliente_id")
        ["importe_total"]
        .shift(1)
    )

    dataset["subida_factura"] = (
        dataset["importe_total"]
        -
        dataset["importe_mes_anterior"]
    )


if (
    "cliente_id" in dataset.columns
    and
    "dias_retraso_pago" in dataset.columns
):

    dataset["media_retraso_3m"] = (

        dataset

        .groupby("cliente_id")

        ["dias_retraso_pago"]

        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).mean()
        )
    )


if (
    "cliente_id" in dataset.columns
    and
    "impago_flag" in dataset.columns
):

    dataset["impagos_3m"] = (

        dataset

        .groupby("cliente_id")

        ["impago_flag"]

        .transform(
            lambda x:
            x.rolling(
                3,
                min_periods=1
            ).sum()
        )
    )


# ============================================================
# 4. VARIABLES INTERACCIÓN
# ============================================================

if (
    "dias_retraso_pago" in dataset.columns
    and
    "importe_total" in dataset.columns
):

    dataset["riesgo_pago"] = (
        dataset["dias_retraso_pago"]
        *
        dataset["importe_total"]
    )


if (
    "impago_flag" in dataset.columns
    and
    "stress_calidad_lag" in dataset.columns
):

    dataset["riesgo_finanzas"] = (
        dataset["impago_flag"]
        *
        dataset["stress_calidad_lag"]
    )


# ============================================================
# 5. ELIMINAR COLUMNAS
# ============================================================

eliminar = [

    "cliente_id",
    "fecha",
    "zona_id",
    "zona_id_x",
    "zona_id_y"

]

dataset = dataset.drop(
    columns=[
        c
        for c
        in eliminar
        if c
        in dataset.columns
    ],
    errors="ignore"
)


# ============================================================
# 6. TARGET
# ============================================================

if "churn" not in dataset.columns:

    raise Exception(
        "No existe columna churn"
    )


X = dataset.drop(
    columns="churn"
)

y = dataset["churn"]


# ============================================================
# 7. VARIABLES CATEGÓRICAS
# ============================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


# ============================================================
# 8. LIMPIEZA
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(
        numeric_only=True
    )
)

X = X.fillna(0)


# ============================================================
# 9. TRAIN TEST
# ============================================================

X_train, X_test, y_train, y_test = (

    train_test_split(

        X,
        y,

        test_size=0.20,

        random_state=42,

        stratify=y

    )

)


# ============================================================
# 10. BALANCEO
# ============================================================

peso = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)

print(
    "scale_pos_weight:",
    round(peso,2)
)


# ============================================================
# 11. MODELO
# ============================================================

modelo = XGBClassifier(

    n_estimators=500,

    learning_rate=0.05,

    max_depth=4,

    min_child_weight=10,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=peso,

    random_state=42,

    eval_metric="logloss",

    n_jobs=-1

)


modelo.fit(
    X_train,
    y_train
)


# ============================================================
# 12. PREDICCIÓN
# ============================================================

proba = (
    modelo
    .predict_proba(
        X_test
    )[:,1]
)


threshold = 0.60


pred = (
    proba
    >=
    threshold
).astype(int)


# ============================================================
# RESULTADOS
# ============================================================

print("\nRESULTADOS")

print(
"Accuracy:",
round(
accuracy_score(
y_test,
pred
)*100,
2
)
)

print(
"Precision:",
round(
precision_score(
y_test,
pred,
zero_division=0
),
4
)
)

print(
"Recall:",
round(
recall_score(
y_test,
pred
),
4
)
)

print(
"F1:",
round(
f1_score(
y_test,
pred
),
4
)
)

print(
"ROC:",
round(
roc_auc_score(
y_test,
proba
),
4
)
)

print(
"PR:",
round(
average_precision_score(
y_test,
proba
),
4
)
)

print(
"\nMatriz:"
)

print(
confusion_matrix(
y_test,
pred
)
)

print(
"\nClassification Report"
)

print(
classification_report(
y_test,
pred
)
)

Shape inicial: (317579, 35)
scale_pos_weight: 160.41

RESULTADOS
Accuracy: 90.26
Precision: 0.0213
Recall: 0.3274
F1: 0.04
ROC: 0.7078
PR: 0.0237

Matriz:
[[57200  5922]
 [  265   129]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.91      0.95     63122
           1       0.02      0.33      0.04       394

    accuracy                           0.90     63516
   macro avg       0.51      0.62      0.49     63516
weighted avg       0.99      0.90      0.94     63516



Aunque el modelo con una exactitud del 90,26 % presentó un mayor porcentaje de aciertos globales, no fue seleccionado como modelo final debido a que su capacidad para detectar clientes con riesgo de abandono era inferior. En concreto, identificó 129 clientes churn reales, frente a los 148 detectados por la configuración finalmente elegida. Dado que el objetivo principal del estudio es anticipar el abandono y maximizar la detección de clientes con riesgo de churn, se priorizaron métricas como el recall, el F1-score y la capacidad discriminativa del modelo frente a la exactitud global. Por ello, se optó por la configuración con un umbral de decisión de 0,60, que permitió alcanzar un recall del 37,56 % y detectar un mayor número de clientes potencialmente susceptibles de abandonar la compañía, aun a costa de una ligera reducción en la accuracy.


In [17]:
# ============================================================
# MODELO XGBOOST CHURN DEFENDIBLE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df.shape)


# ======================================================
# 2. CREAR VARIABLES DERIVADAS
# ======================================================

df["cliente_moroso"] = np.where(
    df["dias_retraso_pago"] > 0,
    1,
    0
)

df["riesgo_financiero"] = (
    df["impago_flag"].fillna(0) * 5
    + df["dias_retraso_pago"].fillna(0)
)

df["riesgo_consumo"] = (
    df["variacion_consumo_pct"].fillna(0).abs()
)

df["retraso_impago"] = (
    df["dias_retraso_pago"].fillna(0)
    * df["impago_flag"].fillna(0)
)


# ======================================================
# 3. ELIMINAR VARIABLES NO VÁLIDAS
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y"
]

df = df.drop(
    columns=[c for c in quitar if c in df.columns],
    errors="ignore"
)


# ======================================================
# 4. TARGET
# ======================================================

X = df.drop(columns="churn")
y = df["churn"]


# ======================================================
# 5. VARIABLES CATEGÓRICAS
# ======================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


# ======================================================
# 6. LIMPIEZA
# ======================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(numeric_only=True)
)

X = X.fillna(0)


# ======================================================
# 7. TRAIN / TEST
# ======================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.20,
    random_state=42
)


# ======================================================
# 8. PESO CLASE MINORITARIA
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

print("\nscale_pos_weight:", round(peso, 2))


# ======================================================
# 9. MODELO XGBOOST
# ======================================================

modelo = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=20,
    gamma=5,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=5,
    reg_lambda=10,
    scale_pos_weight=peso,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_train, y_train)


# ======================================================
# 10. PROBABILIDADES
# ======================================================

proba = modelo.predict_proba(X_test)[:, 1]


# ======================================================
# 11. TABLA DE THRESHOLDS
# ======================================================

resultados = []

for threshold in np.arange(0.30, 0.81, 0.01):

    pred_temp = (proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        pred_temp
    ).ravel()

    resultados.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_test, pred_temp),
        "precision": precision_score(y_test, pred_temp, zero_division=0),
        "recall": recall_score(y_test, pred_temp, zero_division=0),
        "f1": f1_score(y_test, pred_temp, zero_division=0),
        "clientes_churn_detectados": tp,
        "clientes_churn_no_detectados": fn,
        "falsos_positivos": fp,
        "total_predichos_churn": tp + fp
    })

df_thresholds = pd.DataFrame(resultados)


print("\n" + "="*70)
print("TABLA DE THRESHOLDS ORDENADA POR F1")
print("="*70)

print(
    df_thresholds
    .sort_values(by="f1", ascending=False)
    .head(15)
)


print("\n" + "="*70)
print("TABLA DE THRESHOLDS PARA DETECTAR MÁS CHURN")
print("="*70)

print(
    df_thresholds
    .sort_values(by="clientes_churn_detectados", ascending=False)
    .head(15)
)


# ======================================================
# 12. ELEGIR THRESHOLD FINAL
# ======================================================
# Puedes cambiar este valor:
# 0.70 = más equilibrado
# 0.60 = detecta más churn, pero aumenta falsos positivos
# 0.55 = detecta todavía más, pero con más ruido

threshold_final = 0.60

pred = (proba >= threshold_final).astype(int)


# ======================================================
# 13. RESULTADOS FINALES
# ======================================================

tn, fp, fn, tp = confusion_matrix(
    y_test,
    pred
).ravel()

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred, zero_division=0)
f1 = f1_score(y_test, pred, zero_division=0)
roc = roc_auc_score(y_test, proba)
pr_auc = average_precision_score(y_test, proba)


print("\n" + "="*70)
print("RESULTADO FINAL XGBOOST")
print("="*70)

print("Threshold utilizado:", threshold_final)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(classification_report(y_test, pred, zero_division=0))


# ======================================================
# 14. IMPORTANCIA DE VARIABLES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X.columns,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*70)
print("TOP 20 VARIABLES MÁS IMPORTANTES")
print("="*70)

print(importancias.head(20))

Shape inicial: (317579, 35)

scale_pos_weight: 160.41

TABLA DE THRESHOLDS ORDENADA POR F1
    threshold  accuracy  precision    recall        f1  \
46       0.76  0.966355   0.031200  0.147208  0.051487   
45       0.75  0.961868   0.028372  0.154822  0.047956   
49       0.79  0.976352   0.031303  0.093909  0.046954   
48       0.78  0.973125   0.030064  0.106599  0.046901   
47       0.77  0.969882   0.029138  0.119289  0.046836   
44       0.74  0.957381   0.026994  0.167513  0.046495   
43       0.73  0.952658   0.026459  0.185279  0.046305   
50       0.80  0.979123   0.032129  0.081218  0.046043   
42       0.72  0.948234   0.025262  0.195431  0.044741   
41       0.71  0.942991   0.024462  0.210660  0.043834   
40       0.70  0.937858   0.023854  0.225888  0.043152   
39       0.69  0.932836   0.023387  0.241117  0.042639   
38       0.68  0.926680   0.022192  0.251269  0.040783   
36       0.66  0.914982   0.021232  0.281726  0.039488   
37       0.67  0.920603   0.021215  0.2

Se desarrolló un modelo XGBoost incorporando variables derivadas relacionadas con el comportamiento financiero y de consumo de los clientes. El modelo obtuvo un ROC-AUC de 0,7118 y, utilizando un umbral de 0,60, consiguió detectar 148 de los 394 clientes que realmente abandonaron la compañía, alcanzando un recall del 37,56 %.
Aunque este enfoque permitió incrementar la detección de clientes con riesgo de churn, presentó una precisión muy reducida y un elevado número de falsos positivos (7.869), lo que podría generar costes innecesarios en campañas de retención.
Por este motivo, y tras comparar sus resultados con los obtenidos por otros enfoques, se decidió no seleccionar este modelo como modelo final del proyecto, optando por una alternativa que ofrecía un mejor equilibrio entre capacidad predictiva, estabilidad y aplicabilidad en un entorno empresarial real.

In [18]:
# ============================================================
# MODELO FINAL TFM — XGBOOST CHURN RECALL ALTO
# churn_t_plus_1 + split temporal + threshold por F2/recall
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ======================================================
# 1. CARGAR DATASET
# ======================================================

df = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df.shape)


# ======================================================
# 2. FECHA Y ORDEN TEMPORAL
# ======================================================

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

df = df.sort_values(["cliente_id", "fecha"])


# ======================================================
# 3. CREAR TARGET FUTURO
# ======================================================

df["churn_t_plus_1"] = (
    df.groupby("cliente_id")["churn"]
    .shift(-1)
)

df = df.dropna(subset=["churn_t_plus_1"])

df["churn_t_plus_1"] = df["churn_t_plus_1"].astype(int)


# ======================================================
# 4. VARIABLES DERIVADAS
# ======================================================

df["cliente_moroso"] = np.where(
    df["dias_retraso_pago"] > 0,
    1,
    0
)

df["riesgo_financiero"] = (
    df["impago_flag"].fillna(0) * 5
    + df["dias_retraso_pago"].fillna(0)
)

df["riesgo_consumo"] = (
    df["variacion_consumo_pct"].fillna(0).abs()
)

df["retraso_impago"] = (
    df["dias_retraso_pago"].fillna(0)
    * df["impago_flag"].fillna(0)
)

df["subida_factura_flag"] = np.where(
    df["subida_factura"].fillna(0) > 0,
    1,
    0
)

df["bajada_consumo_flag"] = np.where(
    df["variacion_consumo_pct"].fillna(0) < 0,
    1,
    0
)


# ======================================================
# 5. ELIMINAR VARIABLES NO VÁLIDAS / LEAKAGE
# ======================================================

quitar = [
    "cliente_id",
    "fecha",
    "churn",
    "alerta_churn",
    "zona_id",
    "zona_id_x",
    "zona_id_y"
]

df_model = df.drop(
    columns=[c for c in quitar if c in df.columns],
    errors="ignore"
)


# ======================================================
# 6. FEATURES Y TARGET
# ======================================================

X = df_model.drop(columns="churn_t_plus_1")
y = df_model["churn_t_plus_1"]


# ======================================================
# 7. VARIABLES CATEGÓRICAS
# ======================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


# ======================================================
# 8. LIMPIEZA
# ======================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(numeric_only=True)
)

X = X.fillna(0)


# ======================================================
# 9. SPLIT TEMPORAL TRAIN / VALID / TEST
# ======================================================

fechas = df.loc[X.index, "fecha"]

datos = X.copy()
datos["target"] = y.values
datos["fecha"] = fechas.values

datos = datos.sort_values("fecha")

n = len(datos)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = datos.iloc[:train_end]
valid = datos.iloc[train_end:valid_end]
test = datos.iloc[valid_end:]

X_train = train.drop(columns=["target", "fecha"])
y_train = train["target"]

X_valid = valid.drop(columns=["target", "fecha"])
y_valid = valid["target"]

X_test = test.drop(columns=["target", "fecha"])
y_test = test["target"]

print("\nTamaños:")
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)
print("Test:", X_test.shape)

print("\nChurn rate:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Valid:", round(y_valid.mean() * 100, 4), "%")
print("Test:", round(y_test.mean() * 100, 4), "%")


# ======================================================
# 10. PESO CLASE MINORITARIA
# ======================================================

peso = (y_train == 0).sum() / (y_train == 1).sum()

print("\nscale_pos_weight base:", round(peso, 2))
print("scale_pos_weight usado:", round(peso * 2.5, 2))


# ======================================================
# 11. MODELO XGBOOST RECALL ALTO
# ======================================================

modelo = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=5,
    gamma=0.5,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=1,
    reg_lambda=3,
    scale_pos_weight=peso * 2.5,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

modelo.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)


# ======================================================
# 12. PROBABILIDADES VALIDACIÓN
# ======================================================

proba_valid = modelo.predict_proba(X_valid)[:, 1]


# ======================================================
# 13. TABLA DE THRESHOLDS EN VALIDACIÓN
# ======================================================

resultados = []

for threshold in np.arange(0.10, 0.81, 0.01):

    pred_valid = (proba_valid >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_valid,
        pred_valid
    ).ravel()

    resultados.append({
        "threshold": round(threshold, 2),
        "accuracy": accuracy_score(y_valid, pred_valid),
        "precision": precision_score(y_valid, pred_valid, zero_division=0),
        "recall": recall_score(y_valid, pred_valid, zero_division=0),
        "f1": f1_score(y_valid, pred_valid, zero_division=0),
        "f2": fbeta_score(y_valid, pred_valid, beta=2, zero_division=0),
        "clientes_churn_detectados": tp,
        "clientes_churn_no_detectados": fn,
        "falsos_positivos": fp,
        "total_predichos_churn": tp + fp
    })

df_thresholds = pd.DataFrame(resultados)

print("\n" + "="*70)
print("THRESHOLDS ORDENADOS POR RECALL")
print("="*70)

print(
    df_thresholds
    .sort_values(by="recall", ascending=False)
    .head(20)
)

print("\n" + "="*70)
print("THRESHOLDS ORDENADOS POR F2")
print("="*70)

print(
    df_thresholds
    .sort_values(by="f2", ascending=False)
    .head(20)
)


# ======================================================
# 14. ELEGIR THRESHOLD FINAL
# ======================================================
# Objetivo: detectar más churn.
# Pedimos al menos 35% de recall en validación.
# Si no se consigue, usa el threshold con mejor F2.

recall_minimo = 0.35

candidatos = df_thresholds[
    df_thresholds["recall"] >= recall_minimo
]

if len(candidatos) > 0:
    threshold_final = (
        candidatos
        .sort_values(by="f2", ascending=False)
        .iloc[0]["threshold"]
    )
else:
    threshold_final = (
        df_thresholds
        .sort_values(by="f2", ascending=False)
        .iloc[0]["threshold"]
    )

print("\nThreshold final elegido:", threshold_final)


# ======================================================
# 15. EVALUACIÓN FINAL EN TEST
# ======================================================

proba_test = modelo.predict_proba(X_test)[:, 1]

pred_test = (proba_test >= threshold_final).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    pred_test
).ravel()

accuracy = accuracy_score(y_test, pred_test)
precision = precision_score(y_test, pred_test, zero_division=0)
recall = recall_score(y_test, pred_test, zero_division=0)
f1 = f1_score(y_test, pred_test, zero_division=0)
f2 = fbeta_score(y_test, pred_test, beta=2, zero_division=0)
roc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)


print("\n" + "="*70)
print("RESULTADO FINAL XGBOOST — TEST TEMPORAL")
print("="*70)

print("Threshold utilizado:", threshold_final)
print("Accuracy:", round(accuracy * 100, 2))
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("F2 churn:", round(f2, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        pred_test,
        zero_division=0
    )
)


# ======================================================
# 16. IMPORTANCIA DE VARIABLES
# ======================================================

importancias = pd.DataFrame({
    "Variable": X_train.columns,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*70)
print("TOP 20 VARIABLES MÁS IMPORTANTES")
print("="*70)

print(importancias.head(20))

Shape inicial: (317579, 35)

Tamaños:
Train: (215305, 49)
Valid: (46137, 49)
Test: (46137, 49)

Churn rate:
Train: 0.6818 %
Valid: 0.531 %
Test: 0.5137 %

scale_pos_weight base: 145.67
scale_pos_weight usado: 364.16

THRESHOLDS ORDENADOS POR RECALL
    threshold  accuracy  precision    recall        f1        f2  \
0        0.10  0.053124   0.005419  0.971429  0.010778  0.026505   
1        0.11  0.060797   0.005418  0.963265  0.010775  0.026494   
2        0.12  0.069684   0.005446  0.959184  0.010831  0.026628   
3        0.13  0.078462   0.005498  0.959184  0.010934  0.026874   
4        0.14  0.087002   0.005503  0.951020  0.010942  0.026890   
5        0.15  0.096582   0.005561  0.951020  0.011057  0.027168   
6        0.16  0.105945   0.005571  0.942857  0.011076  0.027211   
7        0.17  0.116414   0.005564  0.930612  0.011062  0.027171   
8        0.18  0.126991   0.005558  0.918367  0.011049  0.027133   
9        0.19  0.137829   0.005627  0.918367  0.011186  0.027464   
10 

Tras comparar los distintos modelos desarrollados, se seleccionó finalmente el modelo XGBoost con un umbral de decisión de 0,69, ya que fue el que proporcionó el mejor equilibrio entre capacidad de detección del churn y rendimiento global del sistema.

Aunque algunos modelos alcanzaban una mayor exactitud, estos presentaban una menor capacidad para identificar a los clientes que realmente abandonaban el servicio. Dado que el objetivo principal del estudio era detectar el mayor número posible de clientes con riesgo de baja, se decidió priorizar la métrica recall frente a la accuracy.

El modelo seleccionado obtuvo una accuracy del 85,18% y un recall del 39,66%, siendo capaz de identificar 94 de los 237 clientes que abandonaron el servicio en el conjunto de prueba. Además, mantuvo unos niveles de rendimiento global satisfactorios y un comportamiento más equilibrado que otros modelos analizados.

Por todo ello, se considera que este modelo constituye la alternativa más adecuada para un sistema de detección temprana de churn, permitiendo a la empresa implementar estrategias de retención sobre aquellos clientes con mayor probabilidad de abandono.
